In [1]:
# ==============================================================================
# CELL 1: 
# ==============================================================================
import os
import gc
import warnings
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.INFO)

INPUT_PATH = '/kaggle/input/notebooks/luminhanh/ba-model-prep-data/'
WORKING_PATH = '/kaggle/working/'

print(f"Da thiet lap duong dan doc du lieu tu: {INPUT_PATH}")

Da thiet lap duong dan doc du lieu tu: /kaggle/input/notebooks/luminhanh/ba-model-prep-data/


In [2]:
# ==============================================================================
# CELL 2: 
# ==============================================================================
print("Dang nap du lieu tu Parquet va Pickle...")

X_train = pd.read_parquet(INPUT_PATH + "X_train.parquet")
y_train = pd.read_parquet(INPUT_PATH + "y_train.parquet").values

X_val = pd.read_parquet(INPUT_PATH + "X_val.parquet")
y_val = pd.read_parquet(INPUT_PATH + "y_val.parquet").values

X_test = pd.read_parquet(INPUT_PATH + "X_test.parquet")

cat_features = ['store_nbr', 'item_nbr', 'family', 'class', 'city', 'state', 'type', 'cluster']
for col in cat_features:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype('category')
        X_val[col] = X_val[col].astype('category')
        X_test[col] = X_test[col].astype('category')

if 'perishable' in X_train.columns:
    train_weights = X_train['perishable'].apply(lambda x: 1.25 if x == 1 else 1.0).values
    val_weights = X_val['perishable'].apply(lambda x: 1.25 if x == 1 else 1.0).values
    X_train_model = X_train.drop(columns=['perishable'])
    X_val_model = X_val.drop(columns=['perishable'])
    X_test_model = X_test.drop(columns=['perishable'])
else:
    train_weights = None
    val_weights = None
    X_train_model = X_train
    X_val_model = X_val
    X_test_model = X_test

del X_train, X_val, X_test
gc.collect()

print("Nap du lieu thanh cong!")
print(f"Kich thuoc tap Train: {X_train_model.shape}")
print(f"Kich thuoc tap Val: {X_val_model.shape}")

Dang nap du lieu tu Parquet va Pickle...
Nap du lieu thanh cong!
Kich thuoc tap Train: (1005090, 632)
Kich thuoc tap Val: (167515, 632)


In [3]:
# ==============================================================================
# CELL 3: 
# ==============================================================================
print("="*60)
print("BAT DAU CHAY OPTUNA DE TIM SIEU THAM SO TOI UU")
print("="*60)

dtrain_opt = xgb.DMatrix(X_train_model, label=y_train[:, 0], weight=train_weights, enable_categorical=True)
dval_opt = xgb.DMatrix(X_val_model, label=y_val[:, 0], weight=val_weights, enable_categorical=True)

def objective(trial):
    print(f"\n---> Dang chay Trial {trial.number}...")
    
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'tree_method': 'hist', 
        'device': 'cuda', 
        'max_bin': 128, # Them tham so max_bin = 128 de giam VRAM tieu thu khi tao histogram tren GPU
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 9), # Giam max_depth xuong toi da la 9 de tranh vuot qua dung luong VRAM
        'min_child_weight': trial.suggest_int('min_child_weight', 10, 300),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 100.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 100.0, log=True),
        'random_state': 42
    }
    
    print(f"Tham so Trial {trial.number}: Depth={params['max_depth']}, LR={params['learning_rate']:.4f}, Subsample={params['subsample']:.2f}")
    
    model = xgb.train(
        params,
        dtrain_opt,
        num_boost_round=600, 
        evals=[(dtrain_opt, 'train'), (dval_opt, 'val')],
        early_stopping_rounds=40, 
        verbose_eval=100
    )
    
    # Luu lai score tot nhat truoc khi giai phong model
    best_score = model.best_score
    
    # Xoa model va thu gom rac ngay lap tuc sau khi trial ket thuc de tra lai VRAM
    del model
    gc.collect()
    
    return best_score

study = optuna.create_study(direction='minimize')

# Them gc_after_trial=True de ep Optuna don rac he thong sau moi lan thu nghiem
study.optimize(objective, n_trials=40, gc_after_trial=True) 

best_params = study.best_params
best_params['objective'] = 'reg:squarederror'
best_params['eval_metric'] = 'rmse'
best_params['tree_method'] = 'hist'
best_params['device'] = 'cuda'
best_params['max_bin'] = 128
best_params['random_state'] = 42

print("\n" + "="*60)
print("KET QUA TIM KIEM THAM SO TOI UU TU OPTUNA")
print("="*60)
print(f"Chi so RMSE tot nhat dat duoc: {study.best_value:.5f}\n")

print("Bo tham so (Best Parameters):")
for key, value in best_params.items():
    print(f"   - {key}: {value}")
print("="*60 + "\n")

del dtrain_opt, dval_opt
gc.collect()

BAT DAU CHAY OPTUNA DE TIM SIEU THAM SO TOI UU


[I 2026-05-14 02:35:57,128] A new study created in memory with name: no-name-c3793af8-9d13-4044-a095-252ba5408912



---> Dang chay Trial 0...
Tham so Trial 0: Depth=7, LR=0.0150, Subsample=0.78
[0]	train-rmse:1.04288	val-rmse:1.02855
[100]	train-rmse:0.58403	val-rmse:0.57904
[200]	train-rmse:0.53940	val-rmse:0.53896
[300]	train-rmse:0.53092	val-rmse:0.53410
[400]	train-rmse:0.52734	val-rmse:0.53273
[500]	train-rmse:0.52525	val-rmse:0.53232
[599]	train-rmse:0.52357	val-rmse:0.53207


[I 2026-05-14 02:38:10,211] Trial 0 finished with value: 0.5320558466115096 and parameters: {'learning_rate': 0.01498827567315032, 'max_depth': 7, 'min_child_weight': 98, 'subsample': 0.7783117262206432, 'colsample_bytree': 0.6506321906044128, 'gamma': 7.141535192929684e-07, 'reg_alpha': 0.0015615570498135763, 'reg_lambda': 30.604671176954426}. Best is trial 0 with value: 0.5320558466115096.



---> Dang chay Trial 1...
Tham so Trial 1: Depth=9, LR=0.0195, Subsample=0.77
[0]	train-rmse:1.03956	val-rmse:1.02527
[100]	train-rmse:0.55880	val-rmse:0.55597
[200]	train-rmse:0.53247	val-rmse:0.53412
[300]	train-rmse:0.52730	val-rmse:0.53141
[400]	train-rmse:0.52487	val-rmse:0.53045
[500]	train-rmse:0.52299	val-rmse:0.52999
[599]	train-rmse:0.52142	val-rmse:0.52959


...

[I 2026-05-14 04:00:37,379] Trial 38 finished with value: 0.5298231666840935 and parameters: {'learning_rate': 0.0339392843475169, 'max_depth': 9, 'min_child_weight': 147, 'subsample': 0.8649433439429199, 'colsample_bytree': 0.5222298438134306, 'gamma': 0.01531229364630584, 'reg_alpha': 2.1817086422601677, 'reg_lambda': 32.93547926561118}. Best is trial 11 with value: 0.5289572322019728.



---> Dang chay Trial 39...
Tham so Trial 39: Depth=8, LR=0.0108, Subsample=0.78
[0]	train-rmse:1.04589	val-rmse:1.03153
[100]	train-rmse:0.63140	val-rmse:0.62458
[200]	train-rmse:0.55198	val-rmse:0.54978
[300]	train-rmse:0.53605	val-rmse:0.53661
[400]	train-rmse:0.53036	val-rmse:0.53306
[500]	train-rmse:0.52753	val-rmse:0.53170
[599]	train-rmse:0.52583	val-rmse:0.53100


[I 2026-05-14 04:03:21,091] Trial 39 finished with value: 0.5309979684101733 and parameters: {'learning_rate': 0.010777028101201418, 'max_depth': 8, 'min_child_weight': 111, 'subsample': 0.7816448509829564, 'colsample_bytree': 0.5721164937606706, 'gamma': 0.03871434793111688, 'reg_alpha': 22.23325838284262, 'reg_lambda': 0.003989934671440092}. Best is trial 11 with value: 0.5289572322019728.



KET QUA TIM KIEM THAM SO TOI UU TU OPTUNA
Chi so RMSE tot nhat dat duoc: 0.52896

Bo tham so (Best Parameters):
   - learning_rate: 0.02169846007994346
   - max_depth: 9
   - min_child_weight: 290
   - subsample: 0.8526315845754777
   - colsample_bytree: 0.5008245874454743
   - gamma: 6.937942054525552e-05
   - reg_alpha: 5.470550313829535
   - reg_lambda: 0.49496783439857517
   - objective: reg:squarederror
   - eval_metric: rmse
   - tree_method: hist
   - device: cuda
   - max_bin: 128
   - random_state: 42



0

In [4]:
# ==============================================================================
# CELL 4:
# ==============================================================================
import pickle
import os

print("="*60)
print("BAT DAU HUAN LUYEN CHINH THUC 16 MO HINH VOI CHECKPOINT")
print("="*60)

dtest = xgb.DMatrix(X_test_model, enable_categorical=True)

CHECKPOINT_FILE = WORKING_PATH + 'test_preds_checkpoint.pkl'
test_preds = []
start_day = 0

if os.path.exists(CHECKPOINT_FILE):
    print("PHAT HIEN FILE CHECKPOINT CU!")
    try:
        with open(CHECKPOINT_FILE, 'rb') as f:
            test_preds = pickle.load(f)
        
        start_day = len(test_preds)
        
        if start_day == 16:
            print("Du lieu Checkpoint cho thay ban DA HOAN THANH huan luyen ca 16 ngay.")
            print("Bo qua qua trinh nay va di thang toi CELL 5 (Hau xu ly).")
        elif start_day > 0:
            print(f"Dang khoi phuc qua trinh: Da hoan thanh {start_day}/16 ngay.")
            print(f"He thong se tu dong CHAY TIEP tu Ngay {start_day + 1}...")
    except Exception as e:
        print(f"Loi doc file Checkpoint. Bat buoc huan luyen lai tu Ngay 1. Loi: {e}")
        test_preds = []
        start_day = 0
else:
    print("Khong tim thay Checkpoint. Bat dau huan luyen moi tu Ngay 1.")

for i in range(start_day, 16):
    print(f"\n--- Dang huan luyen XGBoost cho Ngay {i+1}/16 ---")
    
    dtrain = xgb.DMatrix(X_train_model, label=y_train[:, i], weight=train_weights, enable_categorical=True)
    dval = xgb.DMatrix(X_val_model, label=y_val[:, i], weight=val_weights, enable_categorical=True)
    
    model = xgb.train(
        best_params,
        dtrain,
        num_boost_round=1200, 
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=50,
        verbose_eval=100
    )
    
    pred = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
    test_preds.append(pred)
    
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(test_preds, f)
    
    print(f"Da luu du phong (Checkpoint) thanh cong cho Ngay {i+1}")
    
    del dtrain, dval, model
    gc.collect()

if start_day < 16:
    print("\nHOAN TAT HUAN LUYEN 16 MODELS XGBOOST AN TOAN!")

BAT DAU HUAN LUYEN CHINH THUC 16 MO HINH VOI CHECKPOINT
Khong tim thay Checkpoint. Bat dau huan luyen moi tu Ngay 1.

--- Dang huan luyen XGBoost cho Ngay 1/16 ---
[0]	train-rmse:1.03786	val-rmse:1.02361
[100]	train-rmse:0.54943	val-rmse:0.54919
[200]	train-rmse:0.52690	val-rmse:0.53225
[300]	train-rmse:0.52141	val-rmse:0.53027
[400]	train-rmse:0.51784	val-rmse:0.52960
[500]	train-rmse:0.51455	val-rmse:0.52923
[600]	train-rmse:0.51170	val-rmse:0.52899
[700]	train-rmse:0.50895	val-rmse:0.52881
[800]	train-rmse:0.50643	val-rmse:0.52866
[900]	train-rmse:0.50388	val-rmse:0.52853
[1000]	train-rmse:0.50132	val-rmse:0.52845
[1100]	train-rmse:0.49904	val-rmse:0.52840
[1199]	train-rmse:0.49658	val-rmse:0.52836
Da luu du phong (Checkpoint) thanh cong cho Ngay 1

--- Dang huan luyen XGBoost cho Ngay 2/16 ---
[0]	train-rmse:0.98466	val-rmse:0.98096
[100]	train-rmse:0.56476	val-rmse:0.57355
[200]	train-rmse:0.54774	val-rmse:0.56161
[300]	train-rmse:0.54214	val-rmse:0.55959
[400]	train-rmse:0.53801	

In [5]:
# ==============================================================================
# CELL 5
# ==============================================================================
print("BUOC CUOI: Hau xu ly va Xuat file nop bai...")

y_test_pred = np.array(test_preds, dtype=np.float32).transpose()
del test_preds 
gc.collect()

y_test_pred = np.clip(np.expm1(y_test_pred), 0, None)

print("Dang tao cau truc DataFrame de ghep ID...")
X_test_info = pd.read_parquet(INPUT_PATH + "X_test.parquet", columns=['store_nbr', 'item_nbr'])

df_preds = pd.DataFrame(
    y_test_pred, 
    index=pd.MultiIndex.from_frame(X_test_info), 
    columns=pd.date_range("2017-08-16", periods=16)
)
del y_test_pred, X_test_info
gc.collect()

print("Dang bien doi du lieu sang dang Doc (Long Format)...")
df_preds = df_preds.stack().to_frame("unit_sales")
df_preds.index.names = ["store_nbr", "item_nbr", "date"]

print("Dang map ID tu file test.csv goc...")
df_test_kaggle = pd.read_csv(
    INPUT_PATH + 'test.csv', 
    usecols=['id', 'store_nbr', 'item_nbr', 'date'],
    parse_dates=['date']
)
df_test_kaggle.set_index(['store_nbr', 'item_nbr', 'date'], inplace=True)

submission = df_test_kaggle.join(df_preds)

del df_test_kaggle, df_preds
gc.collect()

submission['unit_sales'] = submission['unit_sales'].fillna(0).astype(np.float32)

submission = submission.reset_index()
submission = submission.sort_values('id')

OUTPUT_FILE = WORKING_PATH + 'submission_xgboost_optuna.csv'
submission[['id', 'unit_sales']].to_csv(OUTPUT_FILE, index=False, float_format='%.4f')

print("\n" + "="*60)
print(f"THANH CONG! File nop bai da san sang: {OUTPUT_FILE}")
print("="*60)

BUOC CUOI: Hau xu ly va Xuat file nop bai...
Dang tao cau truc DataFrame de ghep ID...
Dang bien doi du lieu sang dang Doc (Long Format)...
Dang map ID tu file test.csv goc...

THANH CONG! File nop bai da san sang: /kaggle/working/submission_xgboost_optuna.csv
